# Linear Discriminant Analysis (LDA) 

## 1. Instalacion e imports

In [1]:
!pip install -q scikit-learn scipy joblib matplotlib seaborn mlflow

In [2]:
import os
import time
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis  # ← LDA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

SEED       = 42
MEMBER     = "Alan Osorio"
MLFLOW_URI = "http://ec2-52-5-36-177.compute-1.amazonaws.com:5000"
EXPERIMENT = "elongacion"

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")
np.random.seed(SEED)
print("OK - Imports listos. SEED:", SEED)

OK - Imports listos. SEED: 42


## 2. Carga de artefactos

Se reutilizan las matrices sparse TF-IDF y labels generadas en `02_encoding.ipynb`.

In [3]:
os.chdir('/home/ec2-user/SageMaker/Sin norm. alarg.')

X_tr          = load_npz(f'X_tr_{EXPERIMENT}.npz')
X_val         = load_npz(f'X_val_{EXPERIMENT}.npz')
X_train_tfidf = load_npz(f'X_train_tfidf_{EXPERIMENT}.npz')
X_test_tfidf  = load_npz(f'X_test_tfidf_{EXPERIMENT}.npz')

y_tr    = joblib.load(f'y_tr_{EXPERIMENT}.pkl')
y_val   = joblib.load(f'y_val_{EXPERIMENT}.pkl')
y_train = joblib.load(f'y_train_{EXPERIMENT}.pkl')
y_test  = joblib.load(f'y_test_{EXPERIMENT}.pkl')

print('Shapes cargados:')
print(' X_tr         :', X_tr.shape)
print(' X_val        :', X_val.shape)
print(' X_train_tfidf:', X_train_tfidf.shape)
print(' X_test_tfidf :', X_test_tfidf.shape)
print(' y_tr         :', len(y_tr))
print(' y_val        :', len(y_val))
print(' y_train      :', len(y_train))
print(' y_test       :', len(y_test))


Shapes cargados:
  X_tr          : (1088000, 46635)
  X_val         : (272000, 46635)
  X_train_tfidf : (1360000, 46635)
  X_test_tfidf  : (240000, 46635)
  y_tr          : 1088000
  y_val         : 272000
  y_train       : 1360000
  y_test        : 240000


## 3. Muestreo para entrenamiento del Decision Tree

Un `DecisionTreeClassifier` sobre 46k atributos TF-IDF y 1.36M registros es costoso en memoria y tiempo. Se usa una **muestra estratificada** del train/validation y se reduce dimensionalidad con SVD para que el experimento sea ejecutable y reproducible.

In [4]:
# Sin muestreo — dataset completo igual que en DT
X_tr_sample,  y_tr_sample  = X_tr,  np.asarray(y_tr)
X_val_sample, y_val_sample = X_val, np.asarray(y_val)

print('Datos completos para tuning:')
print(' X_tr_sample :', X_tr_sample.shape)
print(' X_val_sample:', X_val_sample.shape)
print(' balance y_tr_sample :', np.bincount(y_tr_sample))
print(' balance y_val_sample:', np.bincount(y_val_sample))

Datos completos para tuning:
  X_tr_sample : (1088000, 46635)
  X_val_sample: (272000, 46635)
  balance y_tr_sample : [543897 544103]
  balance y_val_sample: [135974 136026]


## 4. Busqueda de hiperparametros

Se entrena un pipeline `TruncatedSVD → DecisionTreeClassifier` y se selecciona la mejor configuracion por `f1_macro` en validacion.

### Hiperparametros explorados
| Parametro | Descripcion |
|---|---|
| `n_components` | Dimensiones SVD antes del arbol |
| `max_depth` | Profundidad maxima del arbol — controla overfitting |
| `min_samples_leaf` | Minimo de muestras en hoja — regularizacion |
| `criterion` | Funcion de impureza: `gini` o `entropy` |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#   1. Requiere invertir la matriz de covarianza intra-clase (p×p) → inviable a 46k
#   2. Asume distribución Gaussiana — los datos densos post-SVD se acercan más
#
# Solución: TruncatedSVD reduce a k dimensiones densas antes de pasarle a LDA.
# Pre-computamos SVD con el MÁXIMO de n_components del param_grid (200)
# y hacemos slice para los casos de menos componentes → SVD solo se ajusta 1 vez.
# ─────────────────────────────────────────────────────────────────────────────

MAX_COMPONENTS = 200  # Máximo n_components del param_grid definido abajo

print(f"⏳ Ajustando TruncatedSVD con {MAX_COMPONENTS} componentes (solo 1 vez)...")
t0 = time.time()

svd_shared = TruncatedSVD(
    n_components=MAX_COMPONENTS,
    algorithm='randomized',  # Más rápido que 'arpack' en matrices grandes sparse
    n_iter=2,                # 2 iteraciones: suficiente para clasificación binaria
    random_state=SEED,
)

# fit sobre train, transform sobre val → evita data leakage
X_tr_svd  = svd_shared.fit_transform(X_tr_sample)   # (1_088_000, 200) — denso
X_val_svd = svd_shared.transform(X_val_sample)       # (272_000,   200) — denso

print(f"✅ SVD listo en {time.time() - t0:.1f}s")
print(f"   Varianza explicada acumulada ({MAX_COMPONENTS} comps): "
      f"{svd_shared.explained_variance_ratio_.sum():.3f}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Hiperparámetros explorados en LDA:
#
#  n_components (SVD) : dimensiones antes de LDA — más componentes = más info
#  solver             : algoritmo de estimación de LDA
#                       'lsqr'  → mínimos cuadrados, soporta shrinkage
#                       'eigen' → descomposición propia, soporta shrinkage
#  shrinkage          : regularización de la matriz de covarianza
#                       'auto'  → estimador Ledoit-Wolf (recomendado en alta dim)
#                        None   → sin regularización (solo funciona bien si n >> p)
#
# NOTA: solver='svd' (default de sklearn) NO soporta shrinkage → no lo usamos.
# ─────────────────────────────────────────────────────────────────────────────

param_grid = [
    {'n_components': 50,  'solver': 'lsqr',  'shrinkage': 'auto'},
    {'n_components': 100, 'solver': 'lsqr',  'shrinkage': 'auto'},
    {'n_components': 100, 'solver': 'eigen', 'shrinkage': 'auto'},
    {'n_components': 200, 'solver': 'lsqr',  'shrinkage': 'auto'},
    {'n_components': 200, 'solver': 'eigen', 'shrinkage': 'auto'},
    {'n_components': 200, 'solver': 'lsqr',  'shrinkage': None},
]

results_lda = []
best_lda    = None
best_params = None
best_f1     = -1

for i, params in enumerate(param_grid):
    t_iter = time.time()
    k = params['n_components']

    # Slice de las primeras k columnas del SVD pre-computado → instantáneo
    X_tr_k  = X_tr_svd[:,  :k]
    X_val_k = X_val_svd[:, :k]

    lda = LinearDiscriminantAnalysis(
        solver=params['solver'],
        shrinkage=params['shrinkage'],
        # priors=None → LDA estima priors desde los datos (clases balanceadas aquí)
    )
    lda.fit(X_tr_k, y_tr_sample)

    y_val_pred  = lda.predict(X_val_k)
    y_val_proba = lda.predict_proba(X_val_k)[:, 1]

    metrics_row = {
        **params,
        'accuracy':        round(accuracy_score(y_val_sample,  y_val_pred),                  4),
        'f1_macro':        round(f1_score(y_val_sample,        y_val_pred, average='macro'), 4),
        'precision_macro': round(precision_score(y_val_sample, y_val_pred, average='macro'), 4),
        'recall_macro':    round(recall_score(y_val_sample,    y_val_pred, average='macro'), 4),
        'roc_auc':         round(roc_auc_score(y_val_sample,   y_val_proba),                 4),
    }
    results_lda.append(metrics_row)

    if metrics_row['f1_macro'] > best_f1:
        best_f1     = metrics_row['f1_macro']
        best_params = params
        best_lda    = lda

    print(f"  [{i+1}/{len(param_grid)}] n_comp={k:3d} | solver={params['solver']:5s} "
          f"| shrinkage={str(params['shrinkage']):4s} "
          f"→ f1={metrics_row['f1_macro']:.4f}  ({time.time()-t_iter:.1f}s)")

df_lda = pd.DataFrame(results_lda).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(df_lda)
print(f'\n🏆 Mejores hiperparámetros: {best_params}')
print(f'   Mejor f1_macro validación: {best_f1:.4f}')


## 5. Entrenamiento final

Se reentrena el mejor pipeline usando la union de la muestra de train y validacion.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Re-fiteamos SVD sobre X_train_tfidf (train+val = 1.36M) con los mejores params.
# Es necesario un nuevo SVD porque el input cambia de X_tr a X_train_tfidf.
# ─────────────────────────────────────────────────────────────────────────────

X_train_final = X_train_tfidf         # (1_360_000, 46_635)
y_train_final = np.asarray(y_train)
k_best        = best_params['n_components']

print(f"⏳ Ajustando SVD final ({k_best} comps) sobre {X_train_final.shape}...")
t0 = time.time()

svd_final = TruncatedSVD(
    n_components=k_best,
    algorithm='randomized',
    n_iter=2,
    random_state=SEED,
)
X_train_final_svd = svd_final.fit_transform(X_train_final)
print(f"✅ SVD final listo en {time.time() - t0:.1f}s")

# Entrenamos LDA con los mejores hiperparámetros
lda_final = LinearDiscriminantAnalysis(
    solver=best_params['solver'],
    shrinkage=best_params['shrinkage'],
)
lda_final.fit(X_train_final_svd, y_train_final)

# Re-empaquetamos como Pipeline para compatibilidad con MLflow y joblib
model_final = Pipeline([
    ('svd', svd_final),
    ('lda', lda_final),
])

print('✅ Modelo final entrenado.')
print(f'   Train final shape : {X_train_final.shape}')
print(f'   n_components      : {k_best}')
print(f'   solver            : {best_params["solver"]}')
print(f'   shrinkage         : {best_params["shrinkage"]}')


## 6. Evaluacion final en test

In [ ]:
y_pred  = model_final.predict(X_test_tfidf)
y_proba = model_final.predict_proba(X_test_tfidf)[:, 1]

acc       = accuracy_score(y_test,  y_pred)
f1        = f1_score(y_test,        y_pred, average='macro')
auc       = roc_auc_score(y_test,   y_proba)
precision = precision_score(y_test, y_pred, average='macro')
recall    = recall_score(y_test,    y_pred, average='macro')

print('Métricas en test:')
print(f'  Accuracy  : {acc:.4f}')
print(f'  F1 macro  : {f1:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  ROC AUC   : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Negativo (0)', 'Positivo (1)']
)

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('LDA + TF-IDF/SVD')
plt.tight_layout()
plt.show()


## 7. Guardar modelo y resultados

In [ ]:
joblib.dump(model_final, 'lda_tfidf_svd_model.pkl')

results_final = pd.DataFrame([{
    'modelo':            'LinearDiscriminantAnalysis',
    'encoding':          'TF-IDF + TruncatedSVD',
    'member':            MEMBER,
    'final_train_size':  len(y_train_final),
    'n_components':      best_params['n_components'],
    'solver':            best_params['solver'],
    'shrinkage':         str(best_params['shrinkage']),
    'accuracy':          round(acc,       4),
    'f1_macro':          round(f1,        4),
    'precision_macro':   round(precision, 4),
    'recall_macro':      round(recall,    4),
    'roc_auc':           round(auc,       4),
}])

results_final.to_csv('results_lda_tfidf_svd.csv', index=False)

print('Archivos guardados:')
print('  lda_tfidf_svd_model.pkl     → pipeline SVD + LDA (FULL dataset)')
print('  results_lda_tfidf_svd.csv   → tabla de métricas')
print()
print(results_final.to_string(index=False))


## 8. Registro en MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('Parcial_1_NLP')

with mlflow.start_run(run_name=f'LDA_TFIDF-SVD_{MEMBER}') as run:
    mlflow.set_tags({
        'user':       'Alan Osorio',
        'member':     MEMBER,
        'model_type': 'LinearDiscriminantAnalysis',
        'encoding':   'TF-IDF + TruncatedSVD',
        'dataset':    'Sentiment140Twitter',
    })

    mlflow.log_params({
        'prep_remove_urls':      True,
        'prep_remove_mentions':  True,
        'prep_remove_hashtags':  True,
        'prep_remove_emojis':    True,
        'prep_remove_stopwords': False,
        'prep_lemmatization':    False,
    })

    mlflow.log_params({
        'vec_type':         'TfidfVectorizer',
        'vec_max_features': 50000,
        'vec_min_df':       5,
        'vec_max_df':       0.95,
        'vec_ngram_range':  '(1,1)',
        'vec_sublinear_tf': False,
        'vec_norm':         'None',
    })

    mlflow.log_params({
        'model':             'LinearDiscriminantAnalysis',
        'dim_reduction':     'TruncatedSVD',
        'n_components':      best_params['n_components'],
        'solver':            best_params['solver'],
        'shrinkage':         str(best_params['shrinkage']),
        'seed':              SEED,
        'final_train_size':  len(y_train_final),
        'test_size':         X_test_tfidf.shape[0],
        'vocab_size':        X_train_tfidf.shape[1],
    })

    # Métricas en train
    y_pred_train  = model_final.predict(X_train_final)
    y_proba_train = model_final.predict_proba(X_train_final)[:, 1]
    mlflow.log_metrics({
        'train_accuracy':  round(accuracy_score(y_train_final,  y_pred_train),                  4),
        'train_f1_macro':  round(f1_score(y_train_final,        y_pred_train, average='macro'), 4),
        'train_precision': round(precision_score(y_train_final, y_pred_train, average='macro'), 4),
        'train_recall':    round(recall_score(y_train_final,    y_pred_train, average='macro'), 4),
        'train_roc_auc':   round(roc_auc_score(y_train_final,   y_proba_train),                 4),
    })

    # Métricas en validación (con el mejor modelo del tuning)
    k_val        = best_params['n_components']
    X_val_k_best = X_val_svd[:, :k_val]
    y_pred_val   = best_lda.predict(X_val_k_best)
    y_proba_val  = best_lda.predict_proba(X_val_k_best)[:, 1]
    mlflow.log_metrics({
        'val_accuracy':  round(accuracy_score(y_val_sample,  y_pred_val),                  4),
        'val_f1_macro':  round(f1_score(y_val_sample,        y_pred_val, average='macro'), 4),
        'val_precision': round(precision_score(y_val_sample, y_pred_val, average='macro'), 4),
        'val_recall':    round(recall_score(y_val_sample,    y_pred_val, average='macro'), 4),
        'val_roc_auc':   round(roc_auc_score(y_val_sample,   y_proba_val),                 4),
    })

    # Métricas en test
    mlflow.log_metrics({
        'test_accuracy':  round(acc,       4),
        'test_f1_macro':  round(f1,        4),
        'test_precision': round(precision, 4),
        'test_recall':    round(recall,    4),
        'test_roc_auc':   round(auc,       4),
    })

    mlflow.log_artifact('results_lda_tfidf_svd.csv')
    mlflow.sklearn.log_model(
        model_final,
        artifact_path='model',
        registered_model_name=f'LDA_TFIDF_SVD_{MEMBER}'
    )

    print('=' * 55)
    print(' Run registrado en MLflow')
    print(f' Servidor    : {MLFLOW_URI}')
    print(' Experimento : Parcial_1_NLP')
    print(f' Corrida     : LDA_TFIDF-SVD_{MEMBER}')
    print(f' Run ID      : {run.info.run_id}')
    print(f' Test F1     : {f1:.4f}')
    print(f' Test AUC    : {auc:.4f}')
    print('=' * 55)
